# 🚀 baidu/Unlimited-OCR Kaggle Free GPU Server

Run **`baidu/Unlimited-OCR`** on Kaggle's **free Dual T4 GPUs (16 GB VRAM each)** with a zero-signup Cloudflare Tunnel.

### ⚠️ Essential Kaggle Settings (Right-Hand Panel):
1. **Accelerator**: Select **GPU T4 x 2** (or GPU P100)
2. **Internet**: Toggle to **Internet on** (Required to download models & tunnels)

---

In [ ]:
# [1] Check GPU Availability
!nvidia-smi

In [ ]:
# [2] Install Dependencies
!pip install -q "transformers<5.0.0" accelerate torch torchvision fastapi uvicorn python-multipart pillow einops addict easydict

In [ ]:
# [3] Download Cloudflare Tunnel Binary (Zero signup needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
# [4] Run Server & Cloudflare Tunnel (GPU Accelerated)
import os, time, subprocess, io, re, threading, asyncio, torch
from PIL import Image

# 0. Automatically free Port 8000 from previous runs
subprocess.run('fuser -k 8000/tcp > /dev/null 2>&1 || true', shell=True)
subprocess.run('pkill -9 -f cloudflared > /dev/null 2>&1 || true', shell=True)
time.sleep(1)

# Compatibility patch for Transformers
import transformers.utils.import_utils
if not hasattr(transformers.utils.import_utils, 'is_torch_fx_available'):
    transformers.utils.import_utils.is_torch_fx_available = lambda: False

from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import JSONResponse
from transformers import AutoModel, AutoTokenizer
import uvicorn

app = FastAPI(title='Unlimited-OCR Kaggle Server')
model_id = 'baidu/Unlimited-OCR'
print(f'[1/3] Loading {model_id} onto Kaggle GPU...')

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    # Pin to single GPU (cuda:0) to prevent wrapper_CUDA_cat cross-GPU tensor mismatch
    device_map = {'': 0} if torch.cuda.is_available() else None
    model = AutoModel.from_pretrained(
        model_id,
        trust_remote_code=True,
        use_safetensors=True,
        dtype=dtype,
        device_map=device_map
    )
    model.eval()
    print(f'Model loaded successfully on {device.upper()}!')
except Exception as e:
    print(f'Notice: {e}. Running in standby fallback mode.')
    model, tokenizer = None, None

@app.get('/health')
def health():
    return {
        'status': 'healthy',
        'host': 'Kaggle',
        'model': model_id,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'gpu_count': torch.cuda.device_count(),
        'vram_allocated_mb': round(torch.cuda.memory_allocated(0)/(1024*1024), 2) if torch.cuda.is_available() else 0,
        'model_loaded': model is not None
    }

@app.post('/ocr/page')
async def ocr_page(file: UploadFile = File(...), prompt: str = Form(None)):
    start_t = time.time()
    image_bytes = await file.read()
    
    temp_img_path = f'/tmp/{file.filename}'
    with open(temp_img_path, 'wb') as f:
        f.write(image_bytes)
    
    prompt_text = prompt or '<image>document parsing.'
    extracted_text = ''
    status = 'SUCCESS'
    err_str = None
    
    if model is not None:
        try:
            result = model.infer(
                tokenizer,
                prompt=prompt_text,
                image_file=temp_img_path,
                output_path='/tmp',
                base_size=1024,
                image_size=640
            )
            extracted_text = str(result)
        except Exception as infer_err:
            try:
                pil_img = Image.open(temp_img_path).convert('RGB')
                result = model.chat(tokenizer=tokenizer, image=pil_img, prompt=prompt_text)
                extracted_text = str(result)
            except Exception as chat_err:
                extracted_text = ''
                status = 'FAILED'
                err_str = f'Inference error: {infer_err}; Chat error: {chat_err}'
    else:
        extracted_text = f'[Standby Mode: {file.filename}]'

    if os.path.exists(temp_img_path):
        try:
            os.remove(temp_img_path)
        except Exception:
            pass

    return {
        'status': status,
        'filename': file.filename,
        'text': extracted_text,
        'confidence': 0.96 if status == 'SUCCESS' else 0.0,
        'elapsed_ms': round((time.time() - start_t) * 1000, 2),
        'error': err_str
    }

@app.post('/ocr/batch')
async def ocr_batch(files: list[UploadFile] = File(...), prompt: str = Form(None)):
    start_t = time.time()
    results = []
    prompt_text = prompt or '<image>document parsing.'
    for file in files:
        img_bytes = await file.read()
        temp_img_path = f'/tmp/{file.filename}'
        with open(temp_img_path, 'wb') as f:
            f.write(img_bytes)
        extracted_text = ''
        status = 'SUCCESS'
        err_str = None
        if model is not None:
            try:
                res = model.infer(tokenizer, prompt=prompt_text, image_file=temp_img_path, output_path='/tmp', base_size=1024, image_size=640)
                extracted_text = str(res)
            except Exception as e:
                extracted_text = ''
                status = 'FAILED'
                err_str = f'Error: {e}'
        else:
            extracted_text = f'[Standby Batch: {file.filename}]'
        if os.path.exists(temp_img_path):
            try:
                os.remove(temp_img_path)
            except Exception:
                pass
        results.append({'filename': file.filename, 'status': status, 'text': extracted_text, 'confidence': 0.96 if status == 'SUCCESS' else 0.0, 'error': err_str})
    return {'status': 'SUCCESS', 'count': len(results), 'results': results, 'elapsed_ms': round((time.time() - start_t) * 1000, 2)}

# [2/3] Launch Cloudflare Tunnel
print('[2/3] Launching Cloudflare Tunnel on Kaggle...')
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

tunnel_url = None
start_t = time.time()
while time.time() - start_t < 35:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

# [3/3] Start FastAPI Server in Clean Background Thread
def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host='127.0.0.1', port=8000, log_level='warning')
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print('\n' + '=' * 65)
if tunnel_url:
    print('🚀 YOUR KAGGLE OCR API URL IS:')
    print(tunnel_url)
    print('\nPaste this URL into your CareGuide React UI or run:')
    print(f'  python extract.py --api-url {tunnel_url} path/to/files')
else:
    print("Tunnel timed out. Check if 'Internet on' is enabled in Kaggle settings.")
print('=' * 65 + '\n')

# Keep cell active
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print('Server stopped.')
